# Comparação dos métodos de segmentação no conjunto de teste

Compara o baseline atual salvo (-300 HU, P99.9, RG) com as variantes históricas
normal/fuzzy e RG/FC. O antigo Normal + RG P99.7 não é usado como baseline.

Os runs são fixados abaixo e devem conter os mesmos exames de teste.
As variantes históricas não foram reexecutadas com a configuração atual:
esta é uma comparação de pipelines completos, não uma ablação que isola apenas
o efeito do fuzzy. A melhor variante é descritiva, não uma seleção para ajustar
parâmetros no conjunto de teste.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from utils.project.notebook_env import configure_notebook_environment

    REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
except Exception:
    current = Path.cwd().resolve()
    REPO_ROOT = next(
        path
        for path in [current, *current.parents]
        if (path / "src").exists() and (path / "output").exists()
    )

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.visualization.variant_comparison import (  # noqa: E402
    build_dice_stats_by_variant,
    build_delta_summary_vs_reference,
    load_variant_run,
    plot_ostia_status_by_variant,
    plot_pair_delta_by_image,
    plot_pair_dice_by_image,
)

RESULT_ROOT = REPO_ROOT / "output/segmentation/runs/mid_res/fuzzy_comparison"
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
RESULT_ROOT

## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

## Carregamento dos Resultados

A análise usa o `results_test.csv` de cada variante para as medições por imagem e o `metadata_test.json` adjacente para identificar os métodos de threshold e segmentação arterial.

In [ ]:
PREFERRED_ORDER = [
    "normal_rg",
    "th_fuzzy_rg",
    "normal_fc",
    "th_fuzzy_fc",
]

PRETTY_NAMES = {
    "normal_rg": "Baseline atual P99.9 + RG",
    "th_fuzzy_rg": "Fuzzy threshold + RG",
    "normal_fc": "Normal + FC",
    "th_fuzzy_fc": "Fuzzy threshold + FC",
}

ANALYSIS_SPLIT = "test"

# Caminhos explícitos evitam trocar silenciosamente de baseline ou duplicar runs.
BASELINE_SUMMARY = (
    REPO_ROOT
    / "output/segmentation/runs/mid_res/current_baseline_p99_9"
    / "test/2026-08-06_10-04-22/numeric/results_test.csv"
)
VARIANT_SUMMARIES = {
    "normal_rg": BASELINE_SUMMARY,
    "th_fuzzy_rg": (
        RESULT_ROOT
        / "test/th_fuzzy_rg/2026-06-20_08-33-25"
        / "numeric/results_test.csv"
    ),
    "normal_fc": (
        RESULT_ROOT / "test/normal_fc/2026-06-19_09-20-42" / "numeric/results_test.csv"
    ),
    "th_fuzzy_fc": (
        RESULT_ROOT
        / "test/th_fuzzy_fc/2026-06-20_23-26-14"
        / "numeric/results_test.csv"
    ),
}

In [ ]:
frames, summaries = [], []
reference_ids = None

for variant in PREFERRED_ORDER:
    path = VARIANT_SUMMARIES[variant]
    if path.name != f"results_{ANALYSIS_SPLIT}.csv":
        raise ValueError(f"Split incompatível: {path}")
    frame, summary = load_variant_run(path, repo_root=REPO_ROOT)
    if frame["IMG_ID"].duplicated().any():
        raise ValueError(f"Exames duplicados em {path}")
    image_ids = set(frame["IMG_ID"])
    if reference_ids is not None and image_ids != reference_ids:
        raise ValueError(f"Coorte diferente do baseline: {variant}")
    reference_ids = image_ids
    # A identidade da análise independe do nome da pasta do run.
    frame["folder_variant"] = variant
    frame["variant_label"] = PRETTY_NAMES[variant]
    summary.update(folder_variant=variant, variant_label=PRETTY_NAMES[variant])
    frames.append(frame)
    summaries.append(summary)

results_df = pd.concat(frames, ignore_index=True)
summary_df = pd.DataFrame(summaries)
print(f"Split analisado: {ANALYSIS_SPLIT}")
print(f"Runs carregados: {len(summary_df)}")
print(f"Exames comuns por variante: {len(reference_ids or [])}")
print(f"Baseline: {BASELINE_SUMMARY.relative_to(REPO_ROOT)}")

In [ ]:
display(summary_df)

## Detecção dos Óstios

Aqui o status é separado em: ambos corretos, ambos toleráveis, encontrados porém incorretos e não encontrado/erro.

In [ ]:
plot_ostia_status_by_variant(
    summary_df,
    preferred_order=PREFERRED_ORDER,
    pretty_names=PRETTY_NAMES,
    save_path=None,
)
plt.show()

## Dice Score por Variante

A tabela abaixo resume o Dice arterial por variante. O gráfico em seguida mostra média e mediana para comparação visual.

In [ ]:
dice_plot = summary_df.copy()
dice_plot["variant_label"] = pd.Categorical(
    dice_plot["variant_label"],
    [PRETTY_NAMES.get(name, name) for name in PREFERRED_ORDER],
    ordered=True,
)
dice_plot = dice_plot.sort_values("variant_label")

In [ ]:
dice_stats_df = build_dice_stats_by_variant(results_df, PREFERRED_ORDER)

display(
    dice_stats_df.round(
        {
            "mean_dice": 4,
            "max_dice": 4,
            "min_dice": 4,
            "std_dice": 4,
            "median_dice": 4,
        }
    )
)

In [ ]:
x = np.arange(len(dice_plot))
width = 0.38
fig, ax = plt.subplots(figsize=(12, 5))
mean_bars = ax.bar(
    x - width / 2, dice_plot["mean_dice"], width, label="Média", color="#4c78a8"
)
median_bars = ax.bar(
    x + width / 2, dice_plot["median_dice"], width, label="Mediana", color="#f58518"
)

for bars in [mean_bars, median_bars]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.008,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

ax.set_xticks(x)
ax.set_xticklabels(dice_plot["variant_label"].astype(str), rotation=35, ha="right")
ax.set_ylabel("Dice arterial", fontsize=12)
y_max = max(0.75, float(dice_plot[["mean_dice", "median_dice"]].max().max()) + 0.08)
ax.set_ylim(0, y_max)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Melhor variante versus baseline

A melhor variante é escolhida pelo maior Dice médio entre as alternativas. A
comparação pareada usa `Normal + RG` como baseline e somente IDs presentes nos
dois resultados. O delta é calculado como `melhor variante - baseline`.

In [ ]:
baseline_variant = "normal_rg"
alternative_summary = summary_df.loc[
    summary_df["folder_variant"].astype(str) != baseline_variant
].copy()
BEST_COMPARISON_VARIANT = str(
    alternative_summary.sort_values("mean_dice", ascending=False).iloc[0][
        "folder_variant"
    ]
)

best_vs_baseline_df = build_delta_summary_vs_reference(
    results_df,
    baseline_variant,
    variants=[BEST_COMPARISON_VARIANT],
    pretty_names=PRETTY_NAMES,
)

print(f"Baseline: {PRETTY_NAMES[baseline_variant]}")
print(
    "Melhor variante: "
    f"{PRETTY_NAMES.get(BEST_COMPARISON_VARIANT, BEST_COMPARISON_VARIANT)}"
)
display(
    best_vs_baseline_df.round(
        {
            "mean_delta": 4,
            "median_delta": 4,
            "max_gain": 4,
            "max_loss": 4,
        }
    )
)

In [ ]:
DICE_PANEL_NAMES = {
    baseline_variant: "Baseline",
    BEST_COMPARISON_VARIANT: "Melhor variante",
}
plot_pair_dice_by_image(
    results_df,
    baseline_variant,
    BEST_COMPARISON_VARIANT,
    pretty_names=DICE_PANEL_NAMES,
    save_path=None,
)
plt.show()

In [ ]:
plot_pair_delta_by_image(
    results_df,
    baseline_variant,
    BEST_COMPARISON_VARIANT,
    pretty_names=PRETTY_NAMES,
    save_path=None,
)
plt.show()